# بیس‌لاین دیجیتال FP32 — VGG9 / CIFAR-10، قابل‌مقایسه با V10.5

فقط CIFAR-10 Python batches را به Kaggle اضافه کنید، یک GPU فعال کنید و Run All بزنید. چک‌پوینت و مدل اینورتر لازم نیست. همهٔ لایه‌ها دیجیتال‌اند: Conv/Dense → BN معمولی → ReLU؛ بدون ADC/DAC، tanh اینورتر، محدودیت ولتاژ یا رسانایی. معماری ویژگی‌ها دقیقاً 64,64,128,128,256,256,512,512 → GAP → Dense512 → Dense10 است؛ pooling بعد از کانولوشن‌های 2،4،6 است.

مقایسه بر دادهٔ یکسان، seed=42، split یکسان 45000/5000، ورودی x/255−0.5، crop/flip یکسان، batch=16، FP32، یک GPU، Adam، نرخ‌های 5e−4/2.5e−4/1.25e−4 و patience=4/3/3 است. BN momentum=.90 و epsilon=.001 است. تعداد کل ایپاک‌ها سقف 65 دارد، مطابق تعداد ایپاک‌های اجراشدهٔ V10.5؛ توقف زودهنگام ممکن است تعداد واقعی را متفاوت کند.

بیس‌لاین از صفر آموزش می‌بیند و Front موقت یا QAT ندارد. این یک مقایسهٔ معماری و پروتکل داده/آموزش است، نه برابری تعداد سلول‌های ممریستوری، شروع وزن‌ها یا هزینهٔ محاسبات هر گام. تعداد پارامترها و تعداد ایپاک‌های واقعی گزارش می‌شوند. logits دیجیتال همان خروجی Dense است؛ ADC-code/8 واحدِ logits مدل آنالوگ است و به خروجی دیجیتال اعمال نمی‌شود.

Test رسمی فقط پس از انتخاب نهایی با validation باز می‌شود. مرجع V10.5 داخل نوت‌بوک همان summary ارسال‌شدهٔ شماست؛ با `HW_REFERENCE_PATH` می‌توانید summary یا ZIP نتیجهٔ بعدی را وارد کنید. زمان V10.6، زمان ادامه است؛ برای مقایسهٔ هزینهٔ کل نباید زمان آموزش اولیه حذف شود.

مراجع API: [Keras BatchNormalization](https://keras.io/api/layers/normalization_layers/batch_normalization/) و [TensorFlow determinism](https://www.tensorflow.org/api_docs/python/tf/config/experimental/enable_op_determinism).


**INFO01 timing + کنترل E0.** ابزار زمان مانند INFO01 است: warm-up/کامپایل روی دیتاست جداگانه با بازگردانی بیت‌به‌بیت، ساخت `tf.data` بیرون از تایمر، همگام‌سازی GPU با خواندن accumulator، و محاسبهٔ «زمان مسیر منتخب» (مجموع زمان خالص هر فاز تا بهترین ایپاک) داخل خود نوت‌بوک.

`PHASES_MODE='reference'` همان بیس‌لاین قبلی است. `PHASES_MODE='phax_endpoint2_matched'` کنترل E0 است: همان برنامهٔ آموزش endpoint2 در PHAX (batch=32، دوازده ایپاک ثابت با 1e-3، یک ایپاک با 2e-4، حداکثر دو ایپاک با 5e-5 و patience=2، بهینه‌ساز تازه در هر فاز). اگر این کنترل هم حدود ۹۰٫۹٪ Validation بدهد، فاصله از برنامهٔ آموزش است، نه از بخش فیزیکی.


In [ ]:
import os, gc, json, math, time, copy, zipfile, hashlib, platform
from pathlib import Path
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL','2')
os.environ.setdefault('TF_DETERMINISTIC_OPS','1')
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

SEED=42
PHASES_MODE='reference'   # 'reference' | 'phax_endpoint2_matched' (control E0)
TIMING_WARMUP_BEFORE_PHASE=True; TIMING_VERIFY_EXACT_RESTORE=True
assert PHASES_MODE in ('reference','phax_endpoint2_matched')
BATCH_SIZE=16 if PHASES_MODE=='reference' else 32; EVAL_BATCH_SIZE=64; STEPS_PER_EXECUTION=16
VAL_SIZE=5000; TOTAL_EPOCH_CAP=65
BN_MOMENTUM=.90; BN_EPSILON=.001
CIFAR_ROOT=None  # optional explicit folder containing data_batch_1 ... test_batch
HW_REFERENCE_PATH=None  # optional summary.json or diagnostics ZIP
if PHASES_MODE=='reference':
    # (label, lr, patience, seed, max_epochs or None=remaining cap); one shared Adam.
    PHASES=(('joint',5e-4,4,SEED+10000,None),('live_bn_low',2.5e-4,3,SEED+12500,None),('live_bn_lower',1.25e-4,3,SEED+14000,None))
    FRESH_OPTIMIZER_PER_PHASE=False
else:
    # PHAX INFO00 endpoint2: 12 fixed @1e-3, 1 @2e-4, <=2 @5e-5 (patience 2); fresh Adam per phase.
    PHASES=(('joint_fixed12',1e-3,12,SEED+12020,12),('refine',2e-4,2,SEED+12070,1),('low',5e-5,2,SEED+12120,2))
    FRESH_OPTIMIZER_PER_PHASE=True
BENCHMARK_REPEATS=30; BENCHMARK_WARMUP=5
np.random.seed(SEED);tf.random.set_seed(SEED)
gpus=tf.config.list_physical_devices('GPU')
if gpus:tf.config.set_visible_devices(gpus[:1],'GPU')
for gpu in gpus[:1]:tf.config.experimental.set_memory_growth(gpu,True)
tf.config.experimental.enable_op_determinism()
tf.keras.mixed_precision.set_global_policy('float32')
RUN_ID=time.strftime('%Y%m%d_%H%M%S')
OUT_DIR=Path('/kaggle/working')/f'phax_digital_baseline_{PHASES_MODE}_{RUN_ID}';OUT_DIR.mkdir(parents=True,exist_ok=True)
RUN_T0=time.perf_counter()
TIMING={k:0. for k in ('data_setup_sec','train_loop_sec','validation_selection_sec','checkpoint_sec','inference_benchmark_sec','final_test_sec','final_train_eval_sec',
                        'compile_warmup_sec','input_pipeline_build_sec')}
HISTORY=[]; PHASE_ROWS=[]; CUM_EPOCH=0
def timed(bucket,fn,*args,**kwargs):
    t=time.perf_counter();out=fn(*args,**kwargs);TIMING[bucket]+=time.perf_counter()-t;return out
ENVIRONMENT={'python':platform.python_version(),'tensorflow':tf.__version__,'keras':tf.keras.__version__,
             'visible_GPU':[tf.config.experimental.get_device_details(g).get('device_name',g.name) for g in gpus[:1]],
             'precision':'float32','batch_size':BATCH_SIZE,'deterministic':True,'multi_GPU_training':False}
print('ENVIRONMENT',ENVIRONMENT)


In [ ]:
def find_cifar10_batches_dir():
    if CIFAR_ROOT is not None:
        p=Path(CIFAR_ROOT)
        if not all((p/f"data_batch_{i}").is_file() for i in range(1,6)) or not (p/"test_batch").is_file(): raise FileNotFoundError("Incomplete explicit CIFAR_ROOT")
        return p
    candidates=[]
    for root in [Path("/kaggle/input"),Path("/mnt/data")]:
        if not root.exists(): continue
        for p in root.rglob("data_batch_1"):
            if all((p.parent/f"data_batch_{i}").exists() for i in range(1,6)) and (p.parent/"test_batch").exists():
                candidates.append(p.parent)
    if not candidates: raise FileNotFoundError("CIFAR-10 Python batches not found")
    return sorted(candidates,key=lambda p:len(str(p)))[0]

def load_cifar_batch(path):
    import pickle
    with open(path,"rb") as f: d=pickle.load(f,encoding="bytes")
    x=d[b"data"].reshape(-1,3,32,32).transpose(0,2,3,1)
    y=np.asarray(d.get(b"labels",d.get(b"fine_labels")),np.int64)
    return x,y

def load_cifar10_train_only():
    root=find_cifar10_batches_dir(); xs=[]; ys=[]
    for i in range(1,6):
        x,y=load_cifar_batch(root/f"data_batch_{i}"); xs.append(x); ys.append(y)
    return np.concatenate(xs),np.concatenate(ys)

def augment_stateless(x,y,seed):
    x=tf.image.resize_with_crop_or_pad(x,40,40)
    x=tf.image.stateless_random_crop(x,[32,32,3],seed)
    x=tf.image.stateless_random_flip_left_right(x,seed+tf.constant([7,13],tf.int32))
    return x,y

def _train_pipeline(x,y,epoch_seed,batch):
    ds=tf.data.Dataset.from_tensor_slices((x,y)).shuffle(len(x),seed=epoch_seed,reshuffle_each_iteration=False)
    ds=ds.enumerate().map(lambda i,xy: augment_stateless(xy[0],xy[1],tf.stack([tf.cast(epoch_seed,tf.int32),tf.cast(i,tf.int32)])),num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

def make_train_ds(epoch_seed,batch):
    return _train_pipeline(x_train,y_train,epoch_seed,batch)

def make_eval_ds(x,y,batch):
    return tf.data.Dataset.from_tensor_slices((x,y)).batch(batch).prefetch(tf.data.AUTOTUNE)
t=time.perf_counter()
x_all,y_all=load_cifar10_train_only()
if len(x_all)!=50000 or not np.array_equal(np.bincount(y_all,minlength=10),np.full(10,5000)):
    raise RuntimeError('Expected the 50000 official CIFAR-10 training images')
x_all=x_all.astype(np.float32)/255.-.5
perm=np.random.default_rng(SEED).permutation(len(x_all));val_idx=perm[:VAL_SIZE];tr_idx=perm[VAL_SIZE:]
x_val,y_val=x_all[val_idx],y_all[val_idx];x_train,y_train=x_all[tr_idx],y_all[tr_idx]
SPLIT_FINGERPRINT=hashlib.sha256(np.asarray(perm,np.int64).tobytes()).hexdigest()
DATASET_SHA256=hashlib.sha256(b''.join((find_cifar10_batches_dir()/f'data_batch_{i}').read_bytes() for i in range(1,6))).hexdigest()
TIMING['data_setup_sec']+=time.perf_counter()-t
print('train/val',len(x_train),len(x_val),'split',SPLIT_FINGERPRINT)


In [ ]:
EMBEDDED_V10_5_REFERENCE={'artifact': 'PHAX_VGG9_CIFAR10_V10_5_STE_PROGRESSIVEBN_2026-10-04', 'run_id': '20261004_100509', 'run_mode': 'from_scratch', 'dataset': 'CIFAR-10', 'seed': 42, 'split_fingerprint': 'c7cee14bf6acb10a103c2d362ed94615d4e67bcf3a944cb2dd10f4d138bccf62', 'vtc_model_sha256': 'd3f0818bf81cc71e987540a7deb1ffd0bade4003c566b79a38fba2a3393020d1', 'vtc_source': 'learned_size1_tanh_surrogate', 'real_HSPICE_LUT_used': False, 'conductance_uS': [0.12, 8.0], 'repeat_policy': 'repeat_off', 'loading_mode': 'loading_off', 'ADC_codes': [-127, 127], 'DAC_codes': [0, 255], 'DAC_volts': [0, 0.5], 'analog_node_limit_volts': 0.5, 'BN_accumulator_bits': 16, 'rounding': 'nearest_ties_to_even', 'logit_temperature_shift': 3, 'float_BN_reference': {'loss': 0.36629091796875, 'accuracy': 0.9086}, 'one_shot_conversion': {'loss': 0.464756640625, 'accuracy': 0.866}, 'validation': {'loss': 0.3607956298828125, 'accuracy': 0.9068}, 'test': {'loss': 0.37230185546875, 'accuracy': 0.9054}, 'train_full': {'loss': 0.03477692057291667, 'accuracy': 0.9889111111111111}, 'optimizer_epochs': 65, 'shiftadd_qat_epochs': 7, 'hardware_cost': {'inverter_instances': 237956, 'load_memristors': 1033678}, 'physical_audit': {'physical_node_max_by_stage': {'1': 0.4857642650604248, '2': 0.48841094970703125, '3': 0.4773867726325989, '4': 0.47940510511398315, '5': 0.4848615825176239, '6': 0.48634591698646545, '7': 0.48783940076828003, '8': 0.48189541697502136, '9': 0.48916342854499817, '10': 0.4845980107784271}, 'logit': {'label': 'final', 'accuracy_train_diagnostic': 1.0, 'median_adc_code_margin': 77.0, 'top1_tie_fraction': 0.0, 'final_ADC_argmax_change_fraction': 0.0, 'crossentropy_T1': 0.0032350923866033554, 'crossentropy_T8': 0.03363418206572533, 'crossentropy_T16': 0.09268604218959808}}, 'max_BN_DAC_code_all_ADC_codes': 255.0, 'timing': {'train_forward_backward_sec': 16756.663870754, 'validation_selection_sec': 1069.8418698129967, 'train_diagnostic_sec': 128.557334582998, 'hardware_codesign_sec': 199.75875602799547, 'physical_audit_sec': 2.972244008000416, 'final_test_sec': 29.3454449689998, 'checkpoint_sec': 329.08786960100605, 'diagnostic_sec': 26.689548567000884, 'total_wall_sec': 18594.965327853}, 'test_used_for_selection': False, 'multi_GPU_training': False, 'single_run_cannot_isolate_each_change': True, 'checkpoint_file': 'PHAX_V10_5_FINAL_CHECKPOINT_20261004_100509.zip'}
def read_hw_reference(path):
    if path is None:return copy.deepcopy(EMBEDDED_V10_5_REFERENCE)
    p=Path(path)
    if p.suffix.lower()=='.zip':
        with zipfile.ZipFile(p) as z:
            hits=[n for n in z.namelist() if Path(n).name=='summary.json']
            if len(hits)!=1:raise ValueError('ZIP must contain exactly one summary.json')
            return json.loads(z.read(hits[0]))
    return json.loads(p.read_text())
HW_REFERENCE=read_hw_reference(HW_REFERENCE_PATH)
if HW_REFERENCE.get('split_fingerprint')!=SPLIT_FINGERPRINT or HW_REFERENCE.get('seed')!=SEED:
    raise RuntimeError('Hardware reference uses another split or seed')
if HW_REFERENCE.get('dataset','CIFAR-10')!='CIFAR-10':raise RuntimeError('Dataset mismatch')
CONFIG={'seed':SEED,'split_fingerprint':SPLIT_FINGERPRINT,'dataset_sha256':DATASET_SHA256,
        'batch_size':BATCH_SIZE,'eval_batch':EVAL_BATCH_SIZE,'total_epoch_cap':TOTAL_EPOCH_CAP,'phases':PHASES,
        'BN_momentum':BN_MOMENTUM,'BN_epsilon':BN_EPSILON,'input_normalization':'x/255-0.5',
        'augmentation':'pad40/stateless_crop32/stateless_horizontal_flip','macro_architecture':'VGG9 8conv+2dense GAP',
        'baseline_logit_temperature':1,'environment':ENVIRONMENT,'front':False,'ADC_DAC':False,
        'phases_mode':PHASES_MODE,'fresh_optimizer_per_phase':FRESH_OPTIMIZER_PER_PHASE,
        'timing_warmup_before_phase':TIMING_WARMUP_BEFORE_PHASE}
(OUT_DIR/'config.json').write_text(json.dumps(CONFIG,indent=2))


In [ ]:
class DigitalVGG9(tf.keras.Model):
    def __init__(self):
        super().__init__(name='digital_vgg9_cifar10_fp32')
        widths=(64,64,128,128,256,256,512,512)
        self.convs=[tf.keras.layers.Conv2D(c,3,padding='same',use_bias=True,name=f'stage{i}',
                    kernel_initializer=tf.keras.initializers.HeNormal(seed=20260910+i-1),bias_initializer='zeros') for i,c in enumerate(widths,1)]
        self.bns=[tf.keras.layers.BatchNormalization(momentum=BN_MOMENTUM,epsilon=BN_EPSILON,name=f'stage{i}_bn') for i in range(1,10)]
        self.fc=tf.keras.layers.Dense(512,kernel_initializer=tf.keras.initializers.HeNormal(seed=20260918),name='stage9')
        self.out=tf.keras.layers.Dense(10,kernel_initializer=tf.keras.initializers.HeNormal(seed=20260919),name='stage10')
    def call(self,x,training=False):
        z=x
        for i,(conv,bn) in enumerate(zip(self.convs,self.bns[:8]),1):
            z=tf.nn.relu(bn(conv(z),training=training))
            if i in (2,4,6):z=tf.nn.max_pool2d(z,2,2,'SAME')
        z=tf.reduce_mean(z,axis=(1,2));z=tf.nn.relu(self.bns[8](self.fc(z),training=training))
        return self.out(z)
model=DigitalVGG9();_=model(tf.convert_to_tensor(x_val[:1]),training=False)
optimizer=tf.keras.optimizers.Adam(PHASES[0][1]);optimizer.build(model.trainable_variables)
loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
print('Parameters:',model.count_params(),'trainable:',sum(int(np.prod(v.shape)) for v in model.trainable_variables))


In [ ]:
_EVAL_CACHE={}
def make_train_step(model,opt):
    acc=tf.Variable(tf.zeros([3],tf.float32),trainable=False)
    @tf.function(reduce_retracing=True)
    def step(x,y):
        with tf.GradientTape() as tape:
            logits=model(x,training=True); loss=loss_fn(y,logits)
        vv=model.trainable_variables; gg=tape.gradient(loss,vv)
        pairs=[(g,v) for g,v in zip(gg,vv) if g is not None]
        if not pairs: raise RuntimeError("all gradients are disconnected")
        opt.apply_gradients(pairs)
        n=tf.cast(tf.shape(y)[0],tf.float32)
        correct=tf.reduce_sum(tf.cast(tf.equal(tf.argmax(logits,-1,output_type=tf.int64),tf.cast(y,tf.int64)),tf.float32))
        return acc.assign_add(tf.stack([loss*n,correct,n]))
    @tf.function(reduce_retracing=True)
    def group(iterator,count):
        def body(i):
            x,y=next(iterator)
            with tf.control_dependencies([step(x,y)]): return [i+1]
        return tf.while_loop(lambda i:i<count,body,[tf.constant(0)],parallel_iterations=1)[0]
    step.accumulator=acc; step.group=group
    return step

def train_epoch(model,step,seed):
    step.accumulator.assign(tf.zeros([3])); ds=make_train_ds(seed,BATCH_SIZE); it=iter(ds)
    count=int(math.ceil(len(x_train)/BATCH_SIZE))
    for j in range(0,count,STEPS_PER_EXECUTION):
        step.group(it,tf.constant(min(STEPS_PER_EXECUTION,count-j),tf.int32))
    ls,correct,n=step.accumulator.numpy().astype(np.float64)
    return {"loss":float(ls/n),"accuracy":float(correct/n)}

def train_epoch_timed(model,step,seed):
    t0=time.perf_counter()
    step.accumulator.assign(tf.zeros([3])); ds=make_train_ds(seed,BATCH_SIZE); it=iter(ds)
    count=int(math.ceil(len(x_train)/BATCH_SIZE)); build=time.perf_counter()-t0
    t1=time.perf_counter()
    for j in range(0,count,STEPS_PER_EXECUTION):
        step.group(it,tf.constant(min(STEPS_PER_EXECUTION,count-j),tf.int32))
    ls,correct,n=step.accumulator.numpy().astype(np.float64)   # host read = GPU sync
    return {"loss":float(ls/n),"accuracy":float(correct/n)},{"pipeline_build_sec":build,"train_sec":time.perf_counter()-t1}

def warmup_classification_step(model,opt,step):
    if not TIMING_WARMUP_BEFORE_PHASE: return
    mv=list(model.variables); ov=list(opt.variables)
    sm=[v.numpy().copy() for v in mv]; so=[v.numpy().copy() for v in ov]; sa=step.accumulator.numpy().copy()
    rem=len(x_train)%BATCH_SIZE; n=BATCH_SIZE+rem
    step.group(iter(_train_pipeline(x_train[:n],y_train[:n],0,BATCH_SIZE)),tf.constant(2 if rem else 1,tf.int32)); _=step.accumulator.numpy()
    for v,a in zip(mv,sm): v.assign(a)
    for v,a in zip(ov,so): v.assign(a)
    step.accumulator.assign(sa)
    if TIMING_VERIFY_EXACT_RESTORE and not (all(np.array_equal(v.numpy(),a) for v,a in zip(mv,sm)) and all(np.array_equal(v.numpy(),a) for v,a in zip(ov,so))):
        raise RuntimeError('warm-up restore is not bit-exact')

def _make_eval(model):
    acc=tf.Variable(tf.zeros([3],tf.float32),trainable=False)
    @tf.function(reduce_retracing=True)
    def step(x,y):
        logits=model(x,training=False)
        ls=tf.keras.losses.sparse_categorical_crossentropy(y,logits,from_logits=True)
        c=tf.reduce_sum(tf.cast(tf.equal(tf.argmax(logits,-1,output_type=tf.int64),tf.cast(y,tf.int64)),tf.float32))
        return acc.assign_add(tf.stack([tf.reduce_sum(ls),c,tf.cast(tf.shape(y)[0],tf.float32)]))
    step.accumulator=acc
    return step

def evaluate(model,x,y,batch=EVAL_BATCH_SIZE):
    key=id(model)
    if key not in _EVAL_CACHE: _EVAL_CACHE[key]=_make_eval(model)
    st=_EVAL_CACHE[key]; st.accumulator.assign(tf.zeros([3]))
    for xb,yb in make_eval_ds(x,y,batch): st(xb,yb)
    ls,c,n=st.accumulator.numpy().astype(np.float64)
    return {"loss":float(ls/n),"accuracy":float(c/n)}


def better(a,b):
    return a['accuracy']>b['accuracy']+1e-12 or (abs(a['accuracy']-b['accuracy'])<=1e-12 and a['loss']<b['loss'])
def snapshot(model,opt):return {'weights':[v.numpy().copy() for v in model.variables],'optimizer':[v.numpy().copy() for v in opt.variables]}
def restore(model,opt,state):
    if len(model.variables)!=len(state['weights']) or len(opt.variables)!=len(state['optimizer']):raise RuntimeError('State mismatch')
    for v,a in zip(model.variables,state['weights']):v.assign(a)
    for v,a in zip(opt.variables,state['optimizer']):v.assign(a)
def save_checkpoint(label,metric):
    t=time.perf_counter();d=OUT_DIR/'checkpoints'/label;d.mkdir(parents=True,exist_ok=True)
    np.savez(d/'weights.npz',**{f'v{i:04d}':v.numpy() for i,v in enumerate(model.variables)})
    np.savez(d/'optimizer.npz',**{f'o{i:04d}':v.numpy() for i,v in enumerate(optimizer.variables)})
    (d/'meta.json').write_text(json.dumps({'architecture':'DigitalVGG9','config':CONFIG,'metric':metric,'epochs_executed':CUM_EPOCH,
          'optimizer_config':tf.keras.optimizers.serialize(optimizer)},indent=2))
    TIMING['checkpoint_sec']+=time.perf_counter()-t;return d
def run_digital_phase(label,lr,patience,seed,max_epochs):
    global CUM_EPOCH
    optimizer.learning_rate.assign(lr);step=make_train_step(model,optimizer)
    best=timed('validation_selection_sec',evaluate,model,x_val,y_val);state=snapshot(model,optimizer);wait=0;used=0
    best_epoch=0;prefix=0.;best_prefix=0.
    if max_epochs>0:timed('compile_warmup_sec',warmup_classification_step,model,optimizer,step)
    for epoch in range(1,max_epochs+1):
        tr,ti=train_epoch_timed(model,step,seed+epoch);train_sec=ti['train_sec'];TIMING['train_loop_sec']+=train_sec
        TIMING['input_pipeline_build_sec']+=ti['pipeline_build_sec'];prefix+=train_sec
        t=time.perf_counter();va=evaluate(model,x_val,y_val);val_sec=time.perf_counter()-t;TIMING['validation_selection_sec']+=val_sec
        CUM_EPOCH+=1;used+=1
        row={'epoch':CUM_EPOCH,'phase':label,'local_epoch':epoch,'lr':lr,'train_loss':tr['loss'],'train_accuracy':tr['accuracy'],
             'val_loss':va['loss'],'val_accuracy':va['accuracy'],'train_epoch_sec':train_sec,'validation_sec':val_sec}
        HISTORY.append(row);pd.DataFrame(HISTORY).to_csv(OUT_DIR/'evaluation_history.csv',index=False);print('EVAL',row,flush=True)
        if better(va,best):best=dict(va);state=snapshot(model,optimizer);wait=0;best_epoch=epoch;best_prefix=prefix;save_checkpoint(label+'_best',best)
        else:wait+=1
        if wait>=patience:break
    restore(model,optimizer,state);save_checkpoint(label+'_best',best)
    PHASE_ROWS.append({'phase':label,'epochs_executed':used,'best_local_epoch':best_epoch,
                       'train_to_best_sec':best_prefix,'actual_training_sec':prefix,**best});return best


In [ ]:
# All selection is validation-only. The official test has not been loaded.
for label,lr,patience,seed,cap in PHASES:
    remaining=TOTAL_EPOCH_CAP-CUM_EPOCH
    if remaining<=0:break
    if FRESH_OPTIMIZER_PER_PHASE:
        optimizer=tf.keras.optimizers.Adam(lr);optimizer.build(model.trainable_variables)
    run_digital_phase(label,lr,patience,seed,remaining if cap is None else min(cap,remaining))
final_val=timed('validation_selection_sec',evaluate,model,x_val,y_val)
final_checkpoint=save_checkpoint('final',final_val)
print('SELECTED FINAL VALIDATION',final_val,'epochs',CUM_EPOCH)


In [ ]:
def benchmark_inference(model,x,repeats=BENCHMARK_REPEATS,warmup=BENCHMARK_WARMUP):
    rows=[]
    for batch in (1,64):
        xb=tf.convert_to_tensor(x[:batch],tf.float32)
        @tf.function
        def infer(z):return model(z,training=False)
        for _ in range(warmup):_=infer(xb).numpy()
        samples=[]
        for _ in range(repeats):
            t=time.perf_counter();_=infer(xb).numpy();samples.append(1000*(time.perf_counter()-t))
        rows.append({'batch':batch,'repeats':repeats,'median_batch_ms':float(np.median(samples)),
                     'p90_batch_ms':float(np.percentile(samples,90)),'median_image_ms':float(np.median(samples)/len(xb)),
                     'synchronization':'output.numpy; includes output transfer','execution':'tf.function FP32 simulation'})
    return rows

inference=timed('inference_benchmark_sec',benchmark_inference,model,x_val)
pd.DataFrame(inference).to_csv(OUT_DIR/'inference_benchmark.csv',index=False)
root=find_cifar10_batches_dir();x_test,y_test=load_cifar_batch(root/'test_batch')
assert len(x_test)==10000
x_test=x_test.astype(np.float32)/255.-.5
test_metric=timed('final_test_sec',evaluate,model,x_test,y_test)
train_metric=timed('final_train_eval_sec',evaluate,model,x_train,y_train)
checkpoint_zip=Path('/kaggle/working')/f'PHAX_DIGITAL_BASELINE_CHECKPOINT_{RUN_ID}.zip'
t=time.perf_counter()
with zipfile.ZipFile(checkpoint_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in final_checkpoint.iterdir():
        if p.is_file():z.write(p,p.name)
TIMING['checkpoint_sec']+=time.perf_counter()-t
TIMING['total_wall_sec']=time.perf_counter()-RUN_T0
TIMING['other_setup_reporting_sec']=TIMING['total_wall_sec']-sum(v for k,v in TIMING.items() if k!='total_wall_sec')
selected_path_train_sec=float(sum(r['train_to_best_sec'] for r in PHASE_ROWS))
summary={'artifact':'PHAX_DIGITAL_BASELINE_FP32_VGG9_CIFAR10','run_id':RUN_ID,'dataset':'CIFAR-10','seed':SEED,
         'phases_mode':PHASES_MODE,'selected_path_train_sec':selected_path_train_sec,
         'actual_training_sec':float(TIMING['train_loop_sec']),
         'timing_definition':'pure train-loop sec to each phase best epoch; warm-up/compile, pipeline build, validation excluded',
         'legacy_reference_selected_path_train_sec':2072.356839279,
         'run_mode':'from_scratch','split_fingerprint':SPLIT_FINGERPRINT,'dataset_sha256':DATASET_SHA256,
         'validation':final_val,'test':test_metric,'train_full':train_metric,'optimizer_epochs':CUM_EPOCH,
         'parameters_total':model.count_params(),'parameters_trainable':sum(int(np.prod(v.shape)) for v in model.trainable_variables),
         'timing':TIMING,'inference_benchmark':inference,'environment':ENVIRONMENT,'test_used_for_selection':False,
         'comparison_limits':['macro architecture and data matched; analog differential cells are separate parameters',
             'epoch cap matched to observed V10.5 total; actual early-stop epochs may differ',
             'hardware continuation has historical training cost and extra optimization'],
         'hardware_reference_artifact':HW_REFERENCE['artifact']}
(OUT_DIR/'summary.json').write_text(json.dumps(summary,indent=2))
hw_t=HW_REFERENCE['timing'];historical=HW_REFERENCE.get('historical_V10_5_timing',{})
comparison=pd.DataFrame([
 {'model':'Digital FP32','test_accuracy_pct':100*test_metric['accuracy'],'val_accuracy_pct':100*final_val['accuracy'],
  'current_train_minutes':TIMING['train_loop_sec']/60,'historical_train_minutes':0.,'current_total_minutes':TIMING['total_wall_sec']/60,'epochs':CUM_EPOCH},
 {'model':HW_REFERENCE['artifact'],'test_accuracy_pct':100*HW_REFERENCE['test']['accuracy'],'val_accuracy_pct':100*HW_REFERENCE['validation']['accuracy'],
  'current_train_minutes':hw_t.get('train_forward_backward_sec',hw_t.get('train_loop_sec',0))/60,
  'historical_train_minutes':historical.get('train_forward_backward_sec',0)/60,'current_total_minutes':hw_t['total_wall_sec']/60,
  'epochs':HW_REFERENCE.get('optimizer_epochs')}])
comparison['all_training_minutes']=comparison.current_train_minutes+comparison.historical_train_minutes
comparison.to_csv(OUT_DIR/'digital_vs_hardware.csv',index=False)
pd.DataFrame(PHASE_ROWS).to_csv(OUT_DIR/'phase_summary.csv',index=False)
if HISTORY:
    df=pd.DataFrame(HISTORY);fig,ax=plt.subplots(figsize=(9,4));ax.plot(df.epoch,100*df.train_accuracy,label='Train augmented');ax.plot(df.epoch,100*df.val_accuracy,label='Validation')
    ax.set(xlabel='Epoch',ylabel='Accuracy (%)');ax.grid(alpha=.2);ax.legend();fig.tight_layout();fig.savefig(OUT_DIR/'accuracy_by_epoch.png',dpi=150);plt.close(fig)
results_zip=Path('/kaggle/working')/f'PHAX_DIGITAL_BASELINE_RESULTS_{RUN_ID}.zip'
with zipfile.ZipFile(results_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.iterdir():
        if p.is_file() and p.suffix in ('.csv','.json','.png'):z.write(p,p.name)
print(comparison.to_string(index=False));print('RESULTS',results_zip);print('CHECKPOINT',checkpoint_zip)


برای مقایسه، `PHAX_DIGITAL_BASELINE_RESULTS_*.zip` را ارسال کنید. ستون زمان آموزش، حلقهٔ train شامل دریافت داده، augmentation، tracing اولیه، forward، backward و Adam است. زمان validation و ذخیرهٔ checkpoint جداست. سرعت استنتاج این نوت‌بوک سرعت شبیه‌سازی نرم‌افزاری است و با زمان مدار آنالوگ واقعی یکسان نیست.

پارامترهای کنترل مقایسه را برای اجرای اول تغییر ندهید. برای مقاله، اجرای seedهای بیشتر و بیس‌لاین با بودجهٔ بهینه‌سازی بیشتر را بعد از این مقایسهٔ اولیه انجام می‌دهیم. بیس‌لاین توان بالقوهٔ شبکهٔ دیجیتالِ بدون محدودیت را اندازه می‌گیرد؛ نباید به‌عنوان اجرای دقیقاً یکسان با مسیر Front/QAT معرفی شود.
